# Mistral-7B customer-support fine-tune v2: QLoRA + held-out evaluation

**What this notebook does**
1. Loads the public **Bitext customer-support dataset** (~27K real-style support conversations, 27 intents in 11 categories).
2. Takes a **stratified 2,000-sample training set** and a separate **100-sample held-out test set** (the model never sees the test set).
3. Generates replies with the **base** Mistral-7B-Instruct-v0.2 model first, to get a baseline.
4. Fine-tunes with **4-bit QLoRA** (r=16, alpha=32), tracking the run in **MLflow**.
5. Generates replies with the **fine-tuned** model on the same 100 test questions.
6. Scores both models with **ROUGE-L** and an **LLM-as-judge win rate** (Llama-3.3-70B on Groq, free).
7. Pushes the adapter to Hugging Face.

**Before you start (Colab)**
- Runtime → Change runtime type → **T4 GPU**.
- Click the 🔑 **Secrets** icon in the left sidebar and add two secrets, enabling notebook access for both:
  - `HF_TOKEN`: a Hugging Face **write** token (huggingface.co → Settings → Access Tokens).
  - `GROQ_API_KEY`: your (new, rotated) Groq key from console.groq.com.
- Then **Runtime → Run all**. Total time is about 1.5 to 2 hours, mostly GPU time; you can leave it running.

At the end, cell 11 prints a results block. Copy it and send it to Claude.

In [ ]:
# 1. Install
!pip install -q -U transformers peft accelerate bitsandbytes trl datasets evaluate rouge_score mlflow groq

In [ ]:
# 2. Settings
BASE_MODEL   = "mistralai/Mistral-7B-Instruct-v0.2"
HF_REPO      = "Anikate0/mistral-7b-bitext-support-qlora"   # new repo; the old adapter stays untouched
N_TRAIN      = 2000
N_TEST       = 100
SEED         = 42
MAX_NEW_TOK  = 200
SYSTEM       = "You are a professional, friendly customer support agent. Reply to the customer's message."

import os, json, random, time
import numpy as np, torch
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 3. Data: stratified train / held-out test split by intent
from datasets import load_dataset
import pandas as pd

df = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train").to_pandas()
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print(f"{len(df):,} rows, {df.intent.nunique()} intents, {df.category.nunique()} categories")

per_intent_test = max(1, N_TEST // df.intent.nunique())
test_df  = df.groupby("intent", group_keys=False).head(per_intent_test).head(N_TEST)
rest     = df.drop(test_df.index)
train_df = rest.groupby("intent", group_keys=False).head(N_TRAIN // df.intent.nunique() + 1).head(N_TRAIN)

overlap = len(set(train_df.instruction) & set(test_df.instruction))
print(f"train={len(train_df)}  test={len(test_df)}  exact-text overlap={overlap} (should be ~0)")

In [ ]:
# 4. Prompt format (Mistral-Instruct). The tokenizer adds <s>; we add </s> after the answer.
def prompt_of(customer_msg):
    return f"[INST] {SYSTEM}\n\n{customer_msg} [/INST]"

from datasets import Dataset
train_ds = Dataset.from_pandas(train_df[["instruction", "response"]].reset_index(drop=True)).map(
    lambda r: {"text": f"{prompt_of(r['instruction'])} {r['response']}</s>"})
test_q   = test_df.instruction.tolist()
test_ref = test_df.response.tolist()
print(train_ds[0]["text"][:400])

In [ ]:
# 5. Load the base model in 4-bit (QLoRA)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto",
                                             dtype=torch.float16)  # T4 has no bf16
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.pad_token = tok.eos_token
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

def generate_batch(m, questions, bs=8):
    """Greedy, batched generation (left padding) - deterministic so base vs fine-tuned is fair."""
    tok.padding_side = "left"
    m.eval(); outs = []
    for i in range(0, len(questions), bs):
        enc = tok([prompt_of(q) for q in questions[i:i+bs]], return_tensors="pt", padding=True).to(m.device)
        with torch.no_grad():
            gen = m.generate(**enc, max_new_tokens=MAX_NEW_TOK, do_sample=False, pad_token_id=tok.eos_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"  {min(i+bs, len(questions))}/{len(questions)}", end="\r")
    return [o.strip() for o in outs]

In [ ]:
# 6. Baseline: base model replies on the held-out test set (~15-25 min)
t0 = time.time()
base_out = generate_batch(model, test_q)
print(f"\nbase generation took {(time.time()-t0)/60:.1f} min")
json.dump(base_out, open("base_outputs.json", "w"))
print("Example:\n", test_q[0], "\n->", base_out[0][:300])

In [ ]:
# 7. QLoRA fine-tuning with MLflow tracking (~30-45 min)
import mlflow
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

os.environ["MLFLOW_EXPERIMENT_NAME"] = "mistral-bitext-qlora"
tok.padding_side = "right"
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
for p in model.parameters():  # fp32 trainable weights for fp16 mixed precision on T4
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

args = SFTConfig(
    output_dir="./mistral-bitext-qlora", num_train_epochs=1,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=10,
    fp16=True,                      # T4 GPUs support fp16, not bf16
    logging_steps=10, save_strategy="no", optim="paged_adamw_8bit",
    max_length=512, dataset_text_field="text", report_to="mlflow", seed=SEED)
trainer = SFTTrainer(model=model, train_dataset=train_ds, processing_class=tok, args=args)
train_result = trainer.train()
losses = [l["loss"] for l in trainer.state.log_history if "loss" in l]
print(f"train loss: first logged {losses[0]:.3f} -> last logged {losses[-1]:.3f}")

In [ ]:
# 8. Fine-tuned replies on the SAME held-out questions
t0 = time.time()
ft_out = generate_batch(model, test_q)
print(f"\nfine-tuned generation took {(time.time()-t0)/60:.1f} min")
json.dump({"q": test_q, "ref": test_ref, "base": base_out, "ft": ft_out}, open("generations.json", "w"), indent=1)
print("Example:\n", test_q[0], "\n->", ft_out[0][:300])

In [ ]:
# 9. ROUGE-L: word-sequence overlap with the reference reply (0 to 1)
import evaluate
rouge = evaluate.load("rouge")
rl_base = rouge.compute(predictions=base_out, references=test_ref)["rougeL"]
rl_ft   = rouge.compute(predictions=ft_out,   references=test_ref)["rougeL"]
print(f"ROUGE-L  base={rl_base:.3f}  fine-tuned={rl_ft:.3f}")

In [ ]:
# 10-12. LLM-as-judge win rate, results + MLflow, publish adapter (Groq gpt-oss-120b judge)
!test -f scripts/judge_and_report.py || wget -q -P scripts https://raw.githubusercontent.com/Anikate001/-Fine-tune-Mistral-7B-with-QLoRA/v2-bitext-eval/scripts/judge_and_report.py
%run -i scripts/judge_and_report.py


In [ ]:
# 11. (done in the script above) - download results
from google.colab import files
!zip -qr mlruns.zip mlruns
for f in ["results.json", "generations.json", "mlruns.zip"]:
    files.download(f)


## After it finishes
1. Copy the **results block** from cell 11 and send it to Claude.
2. Put `results.json` and `generations.json` in this repo (the `results/` folder) and commit them. Reviewers trust numbers they can check.